In [9]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import torch.optim as optim
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
scaler_X = MinMaxScaler()
dfs = pd.read_csv("../data/features.csv", index_col="Time_[s]")
targets = pd.read_csv("../data/targets.csv", index_col=False)
# df = dfs[dfs["Experiment_ID"]==2].drop(columns=["Experiment_ID"])
# target = targets[targets["Experiment_ID"]==2].iloc[:, 2:]
col = "Angle[degree]ORDistance[mm]"
bend_columns = [col for col in dfs.columns if 'BEND' in col]
bend_columns.insert(0,'Experiment_ID')
dfs = dfs.loc[:, bend_columns]

scaler = MinMaxScaler(feature_range=(0, 1))
targets[col] = scaler.fit_transform(targets[[col]])

In [19]:
groups_y = []
max_len_y = targets.groupby("Experiment_ID").size().max()
num_target_features = targets.shape[1] - 1  # exclude Experiment_ID

for exp_id, group in targets.groupby("Experiment_ID"):
    values = group.drop(columns=["Experiment_ID"]).values  # (len_group, num_target_features)
    
    # Pad with NaN
    padded = np.full((max_len_y, num_target_features), np.nan)
    padded[:values.shape[0], :] = values
    groups_y.append(padded)

# Final y: (num_experiments, max_len, num_target_features)
y = np.array(groups_y, dtype=float)

# Replace NaN with 0
Y = np.nan_to_num(y, nan=0.0)



groups = []
max_len = dfs.groupby("Experiment_ID").size().max()  # longest experiment
num_features = dfs.shape[1] - 1  # exclude Experiment_ID

for exp_id, group in dfs.groupby("Experiment_ID"):
    features = group.drop(columns=["Experiment_ID"]).values
    
    # Pad with NaN (or zeros) to match max_len
    padded = np.full((max_len, num_features), np.nan)  
    padded[:features.shape[0], :] = features
    groups.append(padded)


X = np.array(groups, dtype=float)
X = np.nan_to_num(X, nan=0.0)[:,:,:]
Y = np.nan_to_num(y, nan=0.0)[:,0:2,1:-1]
print("Shape:", X.shape, Y.shape)  

Shape: (100, 1743, 6) (100, 2, 4)


In [20]:
from sklearn.ensemble import RandomForestRegressor

# X: (num_samples, seq_len, num_features)
# Y: (num_samples, num_angles, 3)
num_samples, seq_len, num_features = X.shape
num_angles = Y.shape[1]
output_size = Y.shape[2]

# Prepare training data
X_rf = []
Y_rf = []

for sample_idx in range(num_samples):
    for angle_idx in range(num_angles):
        x_seq = X[sample_idx].flatten()
        degree = angle_idx / (num_angles - 1)
        x_with_angle = np.append(x_seq, degree)
        X_rf.append(x_with_angle)
        Y_rf.append(Y[sample_idx, angle_idx])

X_rf = np.array(X_rf)
Y_rf = np.array(Y_rf)

In [21]:
from sklearn.ensemble import RandomForestRegressor
import joblib

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    max_features='sqrt',
    n_jobs=-1,
    oob_score=True,
    random_state=42,
    verbose=0
)

rf.fit(X_rf, Y_rf)
joblib.dump(rf, "rf.joblib")

['rf.joblib']

OOB Score (~0.78)

Out-of-bag (OOB) is like an internal cross-validation for Random Forests.

0.78 means the model explains roughly 78% of the variance on unseen samples from the training data.

This is a good sign that your model is not severely overfitting, since it’s close to your validation R².

2️⃣ Validation R² (~0.71)

This is the actual score on a held-out set.

0.71 means the model explains ~71% of the variance for truly unseen samples.

There’s a slight drop from OOB, which is normal. A drop of ~0.05–0.1 is acceptable.

3️⃣ Interpretation

The fact that OOB > Validation R² by a moderate margin indicates the model is generalizing fairly well.

If OOB >> Validation R², that would indicate overfitting.

If OOB << Validation R², that would suggest the OOB estimate might be unstable (rare).

In [22]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from ipywidgets import interact, IntSlider, fixed
import torch

# ----------------- Prediction Error Window Plot -----------------
def plot_window_for_angle(X, Y, rf, sample_idx=0, angle_idx=0,
                          num_angles=None, num_features=None,
                          patch_size=200, stride=200, permute=False):
    _, seq_len, _ = X.shape
    num_patches = (seq_len - patch_size) // stride + 1

    # Flatten input and append angle
    x_input = X[sample_idx].flatten()
    degree = angle_idx / (num_angles - 1)
    x_with_angle = np.append(x_input, degree)
    y_true = Y[sample_idx, angle_idx]

    window_errors = []
    for w in range(num_patches):
        start_w = w * stride * num_features
        end_w = start_w + patch_size * num_features

        if np.all(x_with_angle[start_w:end_w] == 0):
            window_errors.append(np.nan)
            continue

        x_temp = x_with_angle.copy()
        if permute:
            x_temp[start_w:end_w] = np.random.permutation(x_temp[start_w:end_w])

        y_hat = rf.predict(x_temp.reshape(1, -1))[0]
        error = np.linalg.norm(y_true - y_hat)
        window_errors.append(error)

    window_errors = np.array(window_errors)
    if np.all(np.isnan(window_errors)):
        most_accurate_window = None
    else:
        most_accurate_window = np.nanargmin(window_errors) if permute else np.nanargmax(window_errors)

    # ---------- Combined Figure ----------
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

    # Top subplot: features and error curve
    for f in range(num_features):
        ax1.plot(X[sample_idx, :, f], label=f'Feature {f}')

    error_curve = np.full(seq_len, np.nan)
    for w, err in enumerate(window_errors):
        start = w * stride
        end = start + patch_size
        if end > seq_len:
            end = seq_len
        error_curve[start:end] = err

    ax1_twin = ax1.twinx()
    ax1_twin.plot(range(seq_len), error_curve, color='black', linestyle='--', linewidth=2, label='Prediction Error')
    ax1.set_xlabel("Timestep")
    ax1.set_ylabel("Feature value")
    ax1_twin.set_ylabel("Error (L2 norm)")
    ax1.set_title(f"Sample {sample_idx}, Angle {angle_idx} - Window Errors")
    ax1.legend(loc='upper left')
    ax1_twin.legend(loc='upper right')

    # Bottom subplot: true vs predicted
    y_pred = rf.predict(x_with_angle.reshape(1, -1))[0]
    for c in range(Y.shape[2]):
        ax2.bar(c-0.2, y_true[c], width=0.4, label=f'True Channel {c}' if c==0 else "")
        ax2.bar(c+0.2, y_pred[c], width=0.4, label=f'Pred Channel {c}' if c==0 else "")
    ax2.set_xticks(range(Y.shape[2]))
    ax2.set_ylabel("Target value")
    ax2.set_title(f"True vs Predicted for Sample {sample_idx}, Angle {angle_idx}")
    ax2.legend()

    plt.tight_layout()
    plt.show()

# ----------------- Model-Based Window Importance -----------------
def window_model_importance(X, Y, sample_idx, angle_idx, patch_size, stride, alpha=1.0):
    seq_len, num_features = X.shape[1], X.shape[2]
    num_patches = (seq_len - patch_size) // stride + 1
    y_target = Y[sample_idx, angle_idx, 0]
    importance_matrix = []

    for w in range(num_patches):
        start = w * stride
        end = start + patch_size
        if end > seq_len:
            break
        X_window = X[sample_idx, start:end, :]
        y_window = np.full((end-start,), y_target)
        model = Ridge(alpha=alpha)
        try:
            model.fit(X_window, y_window)
            imp = np.abs(model.coef_)
        except:
            imp = np.zeros(num_features)
        importance_matrix.append(imp)

    return np.array(importance_matrix).T

def plot_window_model_importance(X, Y, sample_idx=0, angle_idx=0,
                                 num_features=None, patch_size=50,
                                 stride=20, alpha=10):
    model_importance = window_model_importance(X, Y, sample_idx, angle_idx, patch_size, stride, alpha)
    fig, ax = plt.subplots(figsize=(12, 6))
    im = ax.imshow(model_importance, aspect="auto", origin="lower")
    ax.set_title(f"Model-Based (Ridge) Window Feature Importance (Sample {sample_idx}, Angle {angle_idx})")
    ax.set_xlabel("Window Index")
    ax.set_ylabel("Feature Index")
    fig.colorbar(im, ax=ax, label="Coefficient Magnitude (|weight|)")
    plt.show()

# ----------------- Occlusion Importance -----------------
def window_occlusion_importance(X, Y, sample_idx, angle_idx, rf, patch_size, stride, num_features):
    seq_len = X.shape[1]
    num_windows = (seq_len - patch_size) // stride + 1
    x_input = X[sample_idx].flatten()
    degree = angle_idx / (Y.shape[1] - 1)
    x_with_angle = np.append(x_input, degree).reshape(1, -1)
    y_pred_full = rf.predict(x_with_angle)[0]

    occlusion_errors = []
    for w in range(num_windows):
        start_w = w * stride * num_features
        end_w = start_w + patch_size * num_features
        if np.all(x_with_angle[0, start_w:end_w] == 0):
            occlusion_errors.append(np.nan)
            continue
        x_masked = x_with_angle.copy()
        x_masked[0, start_w:end_w] = 0
        y_pred_masked = rf.predict(x_masked)[0]
        occlusion_errors.append(np.linalg.norm(y_pred_full - y_pred_masked))

    return np.array(occlusion_errors)

def plot_occlusion_importance_with_features(X, Y, rf, sample_idx=0, angle_idx=0,
                                            num_features=None, patch_size=50, stride=20):
    importance_curve = window_occlusion_importance(X, Y, sample_idx, angle_idx, rf, patch_size, stride, num_features)
    seq_len = X.shape[1]
    fig, ax1 = plt.subplots(figsize=(12, 6))
    for f in range(num_features):
        ax1.plot(X[sample_idx, :, f], label=f'Feature {f}')
    error_curve = np.full(seq_len, np.nan)
    for w, err in enumerate(importance_curve):
        start = w * stride
        end = start + patch_size
        if end > seq_len:
            end = seq_len
        error_curve[start:end] = err
    ax2 = ax1.twinx()
    ax2.plot(range(seq_len), error_curve, color='black', linestyle='--', linewidth=2, label='Occlusion Error')
    ax1.set_xlabel("Timestep")
    ax1.set_ylabel("Feature Value")
    ax2.set_ylabel("Prediction Change (L2 Norm)")
    ax1.set_title(f"Sample {sample_idx}, Angle {angle_idx} - Features & Window Occlusion")
    ax1.legend(loc='upper left')
    ax2.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

# ----------------- Noise Importance -----------------
def window_noise_importance(X, Y, sample_idx, angle_idx, rf, patch_size, stride, num_features,
                            noise_std=0.5, n_repeats=5):
    seq_len = X.shape[1]
    num_windows = (seq_len - patch_size) // stride + 1
    x_input = X[sample_idx].flatten()
    degree = angle_idx / (Y.shape[1] - 1)
    x_with_angle = np.append(x_input, degree).reshape(1, -1)
    y_pred_full = rf.predict(x_with_angle)[0]
    importance_scores = []

    for w in range(num_windows):
        start_w = w * stride * num_features
        end_w   = start_w + patch_size * num_features
        if np.all(x_with_angle[0, start_w:end_w] == 0):
            importance_scores.append(np.nan)
            continue
        diffs = []
        for _ in range(n_repeats):
            x_noisy = x_with_angle.copy()
            x_noisy[0, start_w:end_w] += np.random.normal(0, noise_std, size=(end_w - start_w,))
            y_pred_noisy = rf.predict(x_noisy)[0]
            diffs.append(np.linalg.norm(y_pred_full - y_pred_noisy))
        importance_scores.append(np.mean(diffs))
    return np.array(importance_scores)

def plot_noise_importance(X, Y, rf, sample_idx=0, angle_idx=0,
                          num_features=None, patch_size=50, stride=20):
    importance_curve = window_noise_importance(X, Y, sample_idx, angle_idx, rf,
                                               patch_size, stride, num_features,
                                               noise_std=0.3, n_repeats=10)
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(importance_curve, marker="o", color="navy")
    ax.set_title(f"Window Noise Sensitivity (Sample {sample_idx}, Angle {angle_idx})")
    ax.set_xlabel("Window Index")
    ax.set_ylabel("Prediction Change (L2 Norm)")
    plt.show()

# ----------------- Conditional AE Window Importance -----------------
def conditional_ae_window_importance(X, sample_idx, angle_idx, patch_size, stride,
                                     num_features, ae_model, num_angles, device='cpu'):
    seq_len = X.shape[1]
    num_windows = (seq_len - patch_size) // stride + 1
    errors = []
    x_input = X[sample_idx].flatten()
    degree = angle_idx / (num_angles - 1)
    x_tensor = torch.tensor(x_input, dtype=torch.float32).unsqueeze(0).to(device)
    angle_tensor = torch.tensor([[degree]], dtype=torch.float32).to(device)
    with torch.no_grad():
        x_full_recon = ae_model(x_tensor, angle_tensor).cpu().numpy().flatten()

    for w in range(num_windows):
        start_w = w * stride * num_features
        end_w   = start_w + patch_size * num_features
        if np.all(x_input[start_w:end_w] == 0):
            errors.append(np.nan)
            continue
        x_masked = x_input.copy()
        x_masked[start_w:end_w] = 0
        x_masked_tensor = torch.tensor(x_masked, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            x_recon = ae_model(x_masked_tensor, angle_tensor).cpu().numpy().flatten()
        errors.append(np.linalg.norm(x_full_recon[start_w:end_w] - x_recon[start_w:end_w]))

    return np.array(errors)

def plot_window_importance_conditional(X, sample_idx=0, angle_idx=0,
                                       num_features=None, patch_size=50,
                                       stride=20, ae_model=None, num_angles=None,
                                       device='cpu'):
    _, seq_len, _ = X.shape
    window_errors = conditional_ae_window_importance(X, sample_idx, angle_idx, patch_size, stride,
                                                     num_features, ae_model, num_angles, device)
    fig, ax1 = plt.subplots(figsize=(12, 6))
    for f in range(num_features):
        ax1.plot(X[sample_idx, :, f], label=f'Feature {f}')
    error_curve = np.full(seq_len, np.nan)
    for w, err in enumerate(window_errors):
        start = w * stride
        end = start + patch_size
        if end > seq_len:
            end = seq_len
        error_curve[start:end] = err
    ax1_twin = ax1.twinx()
    ax1_twin.plot(range(seq_len), error_curve, color='black', linestyle='--', linewidth=2, label='Window Error')
    ax1.set_xlabel("Timestep")
    ax1.set_ylabel("Feature Value")
    ax1_twin.set_ylabel("Window Reconstruction Error (L2)")
    ax1.set_title(f"Sample {sample_idx}, Angle {angle_idx} - Window Importance")
    ax1.legend(loc='upper left')
    ax1_twin.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

# ----------------- Example: Interactive Widget -----------------
def interactive_widget(func, **kwargs):
    interact(func, **kwargs)


In [23]:
# ----------------- Imports -----------------
from ipywidgets import IntSlider, fixed
# If you saved it as a module:
# from window_analysis import *

# Define required variables
num_angles = Y.shape[1]
num_features = X.shape[2]

# Now call the widget
interactive_widget(
    plot_window_for_angle,
    sample_idx=IntSlider(min=0, max=X.shape[0]-1, step=1, value=0),
    angle_idx=IntSlider(min=0, max=num_angles-1, step=1, value=0),
    X=fixed(X),
    Y=fixed(Y),
    rf=fixed(rf),
    num_angles=fixed(num_angles),
    num_features=fixed(num_features),
    patch_size=IntSlider(min=50, max=500, step=50, value=200),
    stride=IntSlider(min=10, max=200, step=10, value=200),
    permute=fixed(False)
)



interactive(children=(IntSlider(value=0, description='sample_idx', max=99), IntSlider(value=0, description='an…

In [ ]:

# ----------------- Example: Occlusion Importance -----------------
interactive_widget(
    plot_occlusion_importance_with_features,
    sample_idx=IntSlider(min=0, max=X.shape[0]-1, step=1, value=0),
    angle_idx=IntSlider(min=0, max=Y.shape[1]-1, step=1, value=0),
    X=fixed(X),
    Y=fixed(Y),
    rf=fixed(rf),
    num_features=fixed(num_features),
    patch_size=IntSlider(min=20, max=200, step=20, value=50),
    stride=IntSlider(min=5, max=100, step=5, value=20)
)




interactive(children=(IntSlider(value=0, description='sample_idx', max=99), IntSlider(value=0, description='an…

# Occlusion

In [ ]:
# ----------------- Example: Noise Importance -----------------
interactive_widget(
    plot_noise_importance,
    sample_idx=IntSlider(min=0, max=X.shape[0]-1, step=1, value=0),
    angle_idx=IntSlider(min=0, max=Y.shape[1]-1, step=1, value=0),
    X=fixed(X),
    Y=fixed(Y),
    rf=fixed(rf),
    num_features=fixed(num_features),
    patch_size=IntSlider(min=20, max=200, step=20, value=50),
    stride=IntSlider(min=5, max=100, step=5, value=20)
)

interactive(children=(IntSlider(value=0, description='sample_idx', max=99), IntSlider(value=0, description='an…